# Objective Quest DATAQUEST 5.0 — V4 Breakthrough Discovery

This notebook is **not another incremental V1/V2/V3 booster iteration**. It is a staged discovery system built from the evidence of the first three versions:

- V1 public LB: **0.48003**
- V2 public LB: **0.48193**
- V3 did not beat V2 internally
- Current target to beat: **0.61541**

The goal is to cheaply test *genuinely different representations/model families* and kill weak hypotheses before expensive full CV.

### Candidate families
1. **Full raw-categorical CatBoost** — all low-cardinality survey codes treated as true categoricals.
2. **Missing / special-response fingerprinting** — row-level response-pattern representation, not questionnaire-path hand engineering.
3. **Retrieval / nearest-neighbour tabular prediction** — local respondent similarity rather than global tree partitions.
4. **Deep tabular multi-task embedding MLP** — categorical embeddings + continuous inputs + ordinal boundary heads.

A candidate only reaches full 5-fold CV if it wins across multiple untouched probe folds. A Kaggle submission is only produced if the final **unique-ID honest OOF QWK >= 0.61541**.

## 0. Setup & Imports

In [1]:
import os, gc, json, math, random, warnings, time
from pathlib import Path
from dataclasses import dataclass
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import minimize
from scipy.stats import spearmanr, ks_2samp

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import cohen_kappa_score, roc_auc_score, mean_squared_error
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.decomposition import TruncatedSVD
from sklearn.neighbors import NearestNeighbors

from lightgbm import LGBMRegressor, early_stopping, log_evaluation
from catboost import CatBoostRegressor

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print('Environment ready | torch', torch.__version__, '| CUDA', torch.cuda.is_available(), '| GPUs', torch.cuda.device_count())

Environment ready | torch 2.10.0+cu128 | CUDA True | GPUs 2


## 1. Problem Understanding & Discovery Contract

In [2]:
CFG = {
    'VERSION': 'v4_breakthrough_discovery',
    'TARGET': 'life_satisfaction',
    'ID_COL': 'id',
    'HOUSEHOLD_COL': 'household_id',
    'SEED': 42,
    'N_FOLDS': 5,
    'PROBE_FOLDS': [0, 2, 3],  # easy/medium/hard mix; never one-fold promotion again
    'CAL_FOLDS': 5,
    'LOWCARD_MAX_UNIQUE': 120,
    'CATBOOST_ITER': 3500,
    'CATBOOST_PROBE_ITER': 1600,
    'LGB_ITER': 3500,
    'LGB_PROBE_ITER': 1600,
    'EARLY_STOP': 150,
    'LEARNING_RATE': 0.03,
    'CAT_DEVICES': '0:1',
    'RETRIEVAL_MAX_REF_PROBE': 45000,
    'RETRIEVAL_MAX_REF_FULL': 80000,
    'RETRIEVAL_CAT_TOP': 56,
    'RETRIEVAL_CONT_TOP': 14,
    'RETRIEVAL_SVD_DIM': 56,
    'RETRIEVAL_K': 48,
    'DEEP_EPOCHS_PROBE': 10,
    'DEEP_EPOCHS_FULL': 18,
    'DEEP_PATIENCE': 3,
    'DEEP_BATCH': 1024,
    'FINGERPRINT_RP_DIM': 24,
    'PROMOTE_MIN_MEAN_GAIN': 0.015,
    'PROMOTE_MIN_WINS': 3,
    'PROMOTE_ABS_QWK': 0.520,
    'KILL_MAX_MEAN_GAIN': 0.005,
    'SUBMISSION_QWK_GATE': 0.61541,
    'EXPECTED_TRAIN_ROWS': 95609,
    'EXPECTED_TEST_ROWS': 23902,
    'EXPECTED_UNIQUE_TEST_IDS': 23666,
    'V1_PUBLIC_LB': 0.48003,
    'V2_PUBLIC_LB': 0.48193,
    'V1_CROSSFIT': 0.479662,
    'V2_CROSSFIT': 0.481367,
}
OUT = Path('/kaggle/working/oqdq5_v4_discovery')
OUT.mkdir(parents=True, exist_ok=True)
print(CFG)

{'VERSION': 'v4_breakthrough_discovery', 'TARGET': 'life_satisfaction', 'ID_COL': 'id', 'HOUSEHOLD_COL': 'household_id', 'SEED': 42, 'N_FOLDS': 5, 'PROBE_FOLDS': [0, 2, 3], 'CAL_FOLDS': 5, 'LOWCARD_MAX_UNIQUE': 120, 'CATBOOST_ITER': 3500, 'CATBOOST_PROBE_ITER': 1600, 'LGB_ITER': 3500, 'LGB_PROBE_ITER': 1600, 'EARLY_STOP': 150, 'LEARNING_RATE': 0.03, 'CAT_DEVICES': '0:1', 'RETRIEVAL_MAX_REF_PROBE': 45000, 'RETRIEVAL_MAX_REF_FULL': 80000, 'RETRIEVAL_CAT_TOP': 56, 'RETRIEVAL_CONT_TOP': 14, 'RETRIEVAL_SVD_DIM': 56, 'RETRIEVAL_K': 48, 'DEEP_EPOCHS_PROBE': 10, 'DEEP_EPOCHS_FULL': 18, 'DEEP_PATIENCE': 3, 'DEEP_BATCH': 1024, 'FINGERPRINT_RP_DIM': 24, 'PROMOTE_MIN_MEAN_GAIN': 0.015, 'PROMOTE_MIN_WINS': 3, 'PROMOTE_ABS_QWK': 0.52, 'KILL_MAX_MEAN_GAIN': 0.005, 'SUBMISSION_QWK_GATE': 0.61541, 'EXPECTED_TRAIN_ROWS': 95609, 'EXPECTED_TEST_ROWS': 23902, 'EXPECTED_UNIQUE_TEST_IDS': 23666, 'V1_PUBLIC_LB': 0.48003, 'V2_PUBLIC_LB': 0.48193, 'V1_CROSSFIT': 0.479662, 'V2_CROSSFIT': 0.481367}


### Discovery rules

- **No scaling** for tree models. Neural/retrieval candidates may standardize continuous inputs because distance-based and gradient-based models require comparable numerical scales.
- **No SMOTE / oversampling.**
- **No generic tree imputation.** Trees keep native NaN semantics.
- Raw `id` and `household_id` are excluded from model features.
- One probe fold is not enough: promotion requires multi-probe evidence.
- Every probe uses an inner calibration split. Thresholds and blend weights are learned on calibration rows, then evaluated on an untouched outer validation fold.
- If no candidate shows a qualitative jump, the notebook intentionally stops before expensive full CV.

### 1.1 Auto-discover Kaggle input

In [3]:
def discover_data_dir():
    roots = [Path('/kaggle/input/competitions/oq-dq-5'), Path('/kaggle/input')]
    for root in roots:
        if not root.exists():
            continue
        if (root/'train_final.csv').exists() and (root/'test_final.csv').exists():
            return root
        for p in root.rglob('train_final.csv'):
            if (p.parent/'test_final.csv').exists():
                return p.parent
    # local fallback only for portability; Kaggle path is preferred
    for root in [Path('/mnt/data/oqdq5'), Path('/mnt/data/_raw_oq'), Path('.')]:
        if (root/'train_final.csv').exists() and (root/'test_final.csv').exists():
            return root
    raise FileNotFoundError('Could not locate train_final.csv and test_final.csv')

DATA_DIR = discover_data_dir()
train = pd.read_csv(DATA_DIR/'train_final.csv')
test = pd.read_csv(DATA_DIR/'test_final.csv')
dictionary = pd.read_csv(DATA_DIR/'data_dictionary.csv') if (DATA_DIR/'data_dictionary.csv').exists() else pd.DataFrame()
print('DATA_DIR:', DATA_DIR)
print('train', train.shape, 'test', test.shape)
assert len(train) == CFG['EXPECTED_TRAIN_ROWS']
assert len(test) == CFG['EXPECTED_TEST_ROWS']
assert test[CFG['ID_COL']].nunique() == CFG['EXPECTED_UNIQUE_TEST_IDS']

DATA_DIR: /kaggle/input/competitions/oq-dq-5
train (95609, 179) test (23902, 178)


## 2. Preprocessing

### 2.1 Integrity audit and duplicate-safe folds

In [4]:
TARGET, ID_COL, HH_COL = CFG['TARGET'], CFG['ID_COL'], CFG['HOUSEHOLD_COL']
y = train[TARGET].astype(int).to_numpy()
Xraw = train.drop(columns=[TARGET]).copy()
Traw = test.copy()

print('Target distribution')
print(train[TARGET].value_counts(normalize=True).sort_index())
print('Unique train id:', train[ID_COL].nunique(), 'duplicates:', len(train)-train[ID_COL].nunique())
print('Unique test id :', test[ID_COL].nunique(), 'duplicates:', len(test)-test[ID_COL].nunique())

# Exact duplicate-ID rows must stay together.
groups = train[ID_COL].astype(str).to_numpy()
strata = (train[TARGET].astype(str) + '_' + train.get('survey_year', pd.Series('NA', index=train.index)).astype(str)).to_numpy()
sgkf = StratifiedGroupKFold(n_splits=CFG['N_FOLDS'], shuffle=True, random_state=CFG['SEED'])
folds = list(sgkf.split(train, strata, groups))
for f,(tr,va) in enumerate(folds):
    assert set(groups[tr]).isdisjoint(set(groups[va]))
    print(f'fold {f}: train={len(tr):,} val={len(va):,}')

Target distribution
life_satisfaction
1    0.449414
2    0.504022
3    0.037225
4    0.009340
Name: proportion, dtype: float64
Unique train id: 94663 duplicates: 946
Unique test id : 23666 duplicates: 236
fold 0: train=76,498 val=19,111
fold 1: train=76,490 val=19,119
fold 2: train=76,501 val=19,108
fold 3: train=76,458 val=19,151
fold 4: train=76,489 val=19,120


### 2.2 Common helpers: QWK, calibration, honest inner split

In [5]:
def qwk(y_true, y_pred):
    return cohen_kappa_score(np.asarray(y_true, int), np.asarray(y_pred, int), weights='quadratic')

def apply_thresholds(score, thresholds):
    t = np.sort(np.asarray(thresholds, float))
    return np.digitize(np.asarray(score, float), t) + 1

def optimize_thresholds(y_true, score, x0=(1.55,2.03,2.52)):
    y_true = np.asarray(y_true, int); score=np.asarray(score,float)
    def obj(z):
        z=np.sort(z)
        if np.min(np.diff(z)) < 0.04: return 10.0
        return -qwk(y_true, apply_thresholds(score,z))
    starts=[x0,(1.50,2.00,2.45),(1.60,2.10,2.60)]
    best=None
    for s in starts:
        r=minimize(obj, np.array(s,float), method='Nelder-Mead', options={'maxiter':500,'xatol':1e-5,'fatol':1e-6})
        cand=np.sort(r.x); val=-obj(cand)
        if best is None or val>best[0]: best=(val,cand)
    return best[1]

def boundary_auc(y_true, score):
    out={}
    for k in [1,2,3]:
        yy=(np.asarray(y_true)>k).astype(int)
        out[f'auc_gt{k}']=roc_auc_score(yy, score)
    return out

def inner_fit_cal_split(indices, fold_seed):
    sub=train.iloc[indices]
    sub_y=sub[TARGET].astype(str)
    sub_year=sub.get('survey_year', pd.Series('NA',index=sub.index)).astype(str)
    sub_strata=(sub_y+'_'+sub_year).to_numpy()
    sub_groups=sub[ID_COL].astype(str).to_numpy()
    cv=StratifiedGroupKFold(n_splits=CFG['CAL_FOLDS'], shuffle=True, random_state=fold_seed)
    fit_rel, cal_rel = next(cv.split(sub, sub_strata, sub_groups))
    return np.asarray(indices)[fit_rel], np.asarray(indices)[cal_rel]

def tune_blend(y_cal, base_cal, cand_cal, alphas=None):
    if alphas is None: alphas=np.linspace(0,1,21)
    best=(-999,None,None)
    for a in alphas:
        s=(1-a)*base_cal+a*cand_cal
        th=optimize_thresholds(y_cal,s)
        sc=qwk(y_cal,apply_thresholds(s,th))
        if sc>best[0]: best=(sc,float(a),th)
    return {'cal_qwk':best[0], 'alpha':best[1], 'thresholds':best[2]}

### 2.3 Raw feature taxonomy from observed data

In [6]:
EXCLUDE={TARGET,ID_COL,HH_COL}
raw_features=[c for c in Xraw.columns if c not in EXCLUDE]
all_for_card=pd.concat([Xraw[raw_features],Traw[raw_features]],axis=0,ignore_index=True)
card=all_for_card.nunique(dropna=True).sort_values()
lowcard=[c for c in raw_features if card[c] <= CFG['LOWCARD_MAX_UNIQUE']]
continuous=[c for c in raw_features if c not in lowcard]
print('raw features',len(raw_features),'low-card categorical candidates',len(lowcard),'continuous/high-card',len(continuous))
print(card.head(20))

raw features 176 low-card categorical candidates 173 continuous/high-card 3
record_type                   1
adult_sample_flag             1
imputation_number             1
household_respondent_flag     1
multifamily_household_flag    2
adult_status_recode           2
elderly_in_family_flag        2
disability_indicator          3
marketplace_insurance         3
adults_in_family              3
adults_in_household           3
income_imputed_flag           3
no_health_insurance           3
worked_last_week              3
uses_hearing_aid              4
indigenous_health_service     4
uses_mobility_equipment       4
prescription_only_plan        4
children_in_family            4
has_child_gov_insurance       4
dtype: int64


## 3. Fitting for Explainable AI — Stable baseline anchor

In [7]:
def build_tree_frame(df):
    X=df[raw_features].copy()
    for c in X.columns:
        X[c]=pd.to_numeric(X[c],errors='coerce')
    # Keep native NaNs. household/id are already excluded.
    X['missing_count']=df[raw_features].isna().sum(axis=1).astype(np.float32)
    return X

def fit_lgb(train_X, train_y, val_X, val_y, probe=True, seed=42):
    n_est=CFG['LGB_PROBE_ITER'] if probe else CFG['LGB_ITER']
    params=dict(
        objective='regression', n_estimators=n_est, learning_rate=CFG['LEARNING_RATE'],
        num_leaves=31, max_depth=-1, min_child_samples=40,
        subsample=0.9, colsample_bytree=0.85, reg_lambda=1.0, reg_alpha=0.05,
        random_state=seed, n_jobs=-1, verbosity=-1,
    )
    # LightGBM GPU builds differ on Kaggle; CPU fallback is intentionally robust.
    model=LGBMRegressor(**params)
    model.fit(train_X,train_y,eval_set=[(val_X,val_y)],callbacks=[early_stopping(CFG['EARLY_STOP'],verbose=False),log_evaluation(0)])
    return model

# Explainability anchor on one untouched outer fold, for EDA only.
tr0,va0=folds[0]
fit0,cal0=inner_fit_cal_split(tr0,100)
BX=build_tree_frame(Xraw); BT=build_tree_frame(Traw)
anchor=fit_lgb(BX.iloc[fit0],y[fit0],BX.iloc[cal0],y[cal0],probe=True,seed=101)
imp=pd.DataFrame({'feature':BX.columns,'importance':anchor.feature_importances_}).sort_values('importance',ascending=False)
print(imp.head(25).to_string(index=False))

                        feature  importance
          general_health_status         497
           depression_frequency         454
        income_to_poverty_ratio         320
              anxiety_frequency         262
                            age         242
        marital_status_detailed         210
                  weight_pounds         209
                 pseudo_stratum         205
                  survey_weight         194
                     pseudo_psu         194
      worry_about_medical_costs         177
        max_education_in_family         172
              geographic_region         152
                 marital_status         152
   difficulty_social_activities         150
                            sex         145
                  height_inches         138
         difficulty_remembering         135
                  anxiety_level         135
                depression_ever         122
                born_in_country         108
   time_since_last_doctor_visit 

## 4. EDA — Breakthrough Discovery

### 4.1 Categorical signal vs numeric-order signal

In [8]:
def eta_squared(values, target):
    s=pd.Series(values)
    yy=pd.Series(target,index=s.index)
    m=s.notna(); s=s[m].astype(str); yy=yy[m]
    if s.nunique()<2: return 0.0
    gm=yy.mean(); denom=((yy-gm)**2).sum()
    if denom<=0:return 0.0
    means=yy.groupby(s).mean(); counts=s.value_counts()
    num=sum(counts[k]*(means[k]-gm)**2 for k in means.index)
    return float(num/denom)

rows=[]
for c in lowcard:
    vals=Xraw[c]
    eta=eta_squared(vals,y)
    num=pd.to_numeric(vals,errors='coerce')
    mask=num.notna()
    sp=abs(spearmanr(num[mask],y[mask]).correlation) if mask.sum()>50 and num[mask].nunique()>1 else 0.0
    rows.append({'feature':c,'n_unique':int(card[c]),'eta2':eta,'abs_spearman':float(0 if np.isnan(sp) else sp),'cat_minus_ord':eta-float((0 if np.isnan(sp) else sp)**2)})
cat_signal=pd.DataFrame(rows).sort_values(['eta2','cat_minus_ord'],ascending=False)
print(cat_signal.head(30).to_string(index=False))
cat_signal.to_csv(OUT/'categorical_signal_discovery.csv',index=False)

                        feature  n_unique     eta2  abs_spearman  cat_minus_ord
           depression_frequency         8 0.145163      0.322500       0.041156
          general_health_status         7 0.142370      0.361369       0.011783
              anxiety_frequency         8 0.087503      0.259551       0.020137
   difficulty_social_activities         7 0.071398      0.229138       0.018893
                depression_ever         4 0.061165      0.227277       0.009510
         difficulty_remembering         6 0.058169      0.216128       0.011458
         work_limited_by_health         4 0.048504      0.198312       0.009177
       difficulty_errands_alone         6 0.047689      0.190232       0.011501
        income_poverty_category        15 0.046550      0.213499       0.000968
        marital_status_detailed         9 0.044816      0.184169       0.010898
             food_security_4cat         5 0.044335      0.176364       0.013231
       difficulty_walking_steps         

### 4.2 Missing / special-response fingerprint audit

In [9]:
SPECIAL_CODES={7,8,9,97,98,99,110,140,990,996,997,998,999}

def numeric_matrix(df, cols):
    return np.column_stack([pd.to_numeric(df[c],errors='coerce').to_numpy(float) for c in cols])

M=numeric_matrix(Xraw,raw_features)
miss=np.isnan(M)
special=np.isin(M,list(SPECIAL_CODES)) & ~miss
fp_summary=pd.DataFrame({
    'missing_count':miss.sum(1),
    'special_count':special.sum(1),
    'missing_ratio':miss.mean(1),
    'special_ratio':special.mean(1),
})
for c in fp_summary.columns:
    print(c,'Spearman with target =',spearmanr(fp_summary[c],y).correlation)
print('Unique raw missing patterns:',pd.util.hash_pandas_object(pd.DataFrame(miss.astype(np.uint8)),index=False).nunique())

missing_count Spearman with target = 0.09080813549507764
special_count Spearman with target = 0.015545525161957598
missing_ratio Spearman with target = 0.09080813549507764
special_ratio Spearman with target = 0.015545525161957598
Unique raw missing patterns: 94663


### 4.3 Train-test alignment and exact-overlap sanity

In [10]:
print('Mean missing train/test:',Xraw[raw_features].isna().mean().mean(),Traw[raw_features].isna().mean().mean())
# Conservative distribution check on raw continuous/high-card features.
drift=[]
for c in continuous:
    a=pd.to_numeric(Xraw[c],errors='coerce').dropna(); b=pd.to_numeric(Traw[c],errors='coerce').dropna()
    if len(a)>100 and len(b)>100 and a.nunique()>1:
        drift.append((c,ks_2samp(a,b).statistic))
print('Largest numeric KS drift:',sorted(drift,key=lambda x:x[1],reverse=True)[:10])

# Exact overlap after removing identifiers. Prior versions found none; verify again.
base_cols=[c for c in raw_features if c not in {ID_COL,HH_COL}]
tr_hash=pd.util.hash_pandas_object(Xraw[base_cols],index=False)
te_hash=pd.util.hash_pandas_object(Traw[base_cols],index=False)
print('Exact feature-row hash overlap:',len(set(tr_hash).intersection(set(te_hash))))

Mean missing train/test: 0.14406409295815628 0.1441908151466975
Largest numeric KS drift: [('survey_weight', np.float64(0.007341351377696148)), ('income_to_poverty_ratio', np.float64(0.005612561043060682)), ('weight_pounds', np.float64(0.005238475054709091))]
Exact feature-row hash overlap: 0


## 5. Feature Engineering — Candidate Representations

### 5.1 Full raw-categorical CatBoost view

In [11]:
def make_catboost_allcat_frames(fit_idx, other_idx, other_df=None):
    fit_df=Xraw.iloc[fit_idx] if other_df is None else None
    # Build from raw features; all low-card survey codes are categorical, not only a curated top list.
    def transform(df):
        out=df[raw_features].copy()
        for c in lowcard:
            out[c]=out[c].where(out[c].notna(),'__MISSING__').astype(str)
        for c in continuous:
            out[c]=pd.to_numeric(out[c],errors='coerce')
        out['missing_count']=df[raw_features].isna().sum(axis=1).astype(float)
        return out
    return transform

def fit_catboost_allcat(X_fit,y_fit,X_cal,y_cal,probe=True,seed=42):
    cat_cols=[c for c in lowcard if c in X_fit.columns]
    n_iter=CFG['CATBOOST_PROBE_ITER'] if probe else CFG['CATBOOST_ITER']
    base=dict(
        loss_function='RMSE', eval_metric='RMSE', iterations=n_iter,
        learning_rate=CFG['LEARNING_RATE'], depth=7, l2_leaf_reg=5.0,
        random_seed=seed, verbose=False, allow_writing_files=False,
        random_strength=0.4,
    )
    if torch.cuda.is_available():
        try:
            m=CatBoostRegressor(**base,task_type='GPU',devices=CFG['CAT_DEVICES'])
            m.fit(X_fit,y_fit,cat_features=cat_cols,eval_set=(X_cal,y_cal),early_stopping_rounds=CFG['EARLY_STOP'],verbose=False)
            return m
        except Exception as e:
            print('CatBoost GPU fallback:',type(e).__name__,str(e)[:160])
    m=CatBoostRegressor(**base,task_type='CPU',thread_count=-1)
    m.fit(X_fit,y_fit,cat_features=cat_cols,eval_set=(X_cal,y_cal),early_stopping_rounds=CFG['EARLY_STOP'],verbose=False)
    return m

### 5.2 Missing/special-response fingerprint features

In [12]:
FP_RNG=np.random.default_rng(20260830)
MISS_RP=FP_RNG.choice([-1.0,1.0],size=(len(raw_features),CFG['FINGERPRINT_RP_DIM'])).astype(np.float32)
SPEC_RP=FP_RNG.choice([-1.0,1.0],size=(len(raw_features),CFG['FINGERPRINT_RP_DIM'])).astype(np.float32)

def build_fingerprint_features(df):
    mat=numeric_matrix(df,raw_features)
    miss=np.isnan(mat).astype(np.float32)
    spec=(np.isin(mat,list(SPECIAL_CODES)) & ~np.isnan(mat)).astype(np.float32)
    out=pd.DataFrame(index=df.index)
    out['fp_missing_count']=miss.sum(1)
    out['fp_special_count']=spec.sum(1)
    out['fp_missing_ratio']=miss.mean(1)
    out['fp_special_ratio']=spec.mean(1)
    mr=(miss@MISS_RP)/math.sqrt(len(raw_features))
    sr=(spec@SPEC_RP)/math.sqrt(len(raw_features))
    for j in range(CFG['FINGERPRINT_RP_DIM']):
        out[f'fp_miss_rp_{j:02d}']=mr[:,j]
        out[f'fp_spec_rp_{j:02d}']=sr[:,j]
    # Explicit states only for strongest categorical-signal columns, avoiding hundreds of redundant binaries.
    state_cols=cat_signal.head(40)['feature'].tolist()
    for c in state_cols:
        v=pd.to_numeric(df[c],errors='coerce')
        out[f'fp_{c}__missing']=v.isna().astype(np.int8)
        out[f'fp_{c}__special']=v.isin(SPECIAL_CODES).astype(np.int8)
    return out

FPX=build_fingerprint_features(Xraw)
FPT=build_fingerprint_features(Traw)
print('Fingerprint features:',FPX.shape[1])

Fingerprint features: 132


### 5.3 Retrieval representation

In [13]:
def select_retrieval_features(fit_idx):
    yy=y[fit_idx]
    rows=[]
    for c in lowcard:
        rows.append((c,eta_squared(Xraw.iloc[fit_idx][c],yy)))
    cat_sel=[c for c,_ in sorted(rows,key=lambda z:z[1],reverse=True)[:CFG['RETRIEVAL_CAT_TOP']]]
    cont_scores=[]
    for c in continuous:
        v=pd.to_numeric(Xraw.iloc[fit_idx][c],errors='coerce')
        m=v.notna()
        if m.sum()>100 and v[m].nunique()>2:
            r=abs(spearmanr(v[m],yy[m]).correlation)
            if np.isfinite(r): cont_scores.append((c,r))
    cont_sel=[c for c,_ in sorted(cont_scores,key=lambda z:z[1],reverse=True)[:CFG['RETRIEVAL_CONT_TOP']]]
    return cat_sel,cont_sel

def fit_retrieval_encoder(fit_df, cat_cols, cont_cols):
    cat=fit_df[cat_cols].copy()
    for c in cat_cols: cat[c]=cat[c].where(cat[c].notna(),'__MISSING__').astype(str)
    try:
        ohe=OneHotEncoder(handle_unknown='ignore',min_frequency=20,sparse_output=True)
    except TypeError:
        ohe=OneHotEncoder(handle_unknown='ignore',min_frequency=20,sparse=True)
    Z=ohe.fit_transform(cat)
    ncomp=max(8,min(CFG['RETRIEVAL_SVD_DIM'],Z.shape[1]-1 if Z.shape[1]>1 else 1))
    svd=TruncatedSVD(n_components=ncomp,random_state=CFG['SEED'])
    Zcat=svd.fit_transform(Z).astype(np.float32)
    scaler=StandardScaler()
    if cont_cols:
        A=np.column_stack([pd.to_numeric(fit_df[c],errors='coerce').to_numpy(float) for c in cont_cols])
        med=np.nanmedian(A,axis=0); inds=np.where(np.isnan(A)); A[inds]=np.take(med,inds[1])
        Zcont=scaler.fit_transform(A).astype(np.float32)
    else:
        med=np.array([]); Zcont=np.empty((len(fit_df),0),np.float32)
    Zall=np.hstack([Zcat,Zcont]); Zall/=np.maximum(np.linalg.norm(Zall,axis=1,keepdims=True),1e-6)
    return {'ohe':ohe,'svd':svd,'scaler':scaler,'med':med,'cat_cols':cat_cols,'cont_cols':cont_cols},Zall

def transform_retrieval(enc,df):
    cat=df[enc['cat_cols']].copy()
    for c in enc['cat_cols']: cat[c]=cat[c].where(cat[c].notna(),'__MISSING__').astype(str)
    Zcat=enc['svd'].transform(enc['ohe'].transform(cat)).astype(np.float32)
    if enc['cont_cols']:
        A=np.column_stack([pd.to_numeric(df[c],errors='coerce').to_numpy(float) for c in enc['cont_cols']])
        inds=np.where(np.isnan(A)); A[inds]=np.take(enc['med'],inds[1])
        Zcont=enc['scaler'].transform(A).astype(np.float32)
    else: Zcont=np.empty((len(df),0),np.float32)
    Z=np.hstack([Zcat,Zcont]); Z/=np.maximum(np.linalg.norm(Z,axis=1,keepdims=True),1e-6)
    return Z

def retrieval_predict(Zref,yref,Zq,k,temperature=0.12):
    nnm=NearestNeighbors(n_neighbors=min(k,len(Zref)),metric='cosine',algorithm='brute',n_jobs=-1)
    nnm.fit(Zref)
    d,idx=nnm.kneighbors(Zq,return_distance=True)
    w=np.exp(-d/max(temperature,1e-4))
    return (w*yref[idx]).sum(1)/np.maximum(w.sum(1),1e-9)

### 5.4 Deep tabular multi-task embedding model

In [14]:
class TabDataset(Dataset):
    def __init__(self, cats, cont, target=None):
        self.cats=torch.as_tensor(cats,dtype=torch.long)
        self.cont=torch.as_tensor(cont,dtype=torch.float32)
        self.target=None if target is None else torch.as_tensor(target,dtype=torch.float32)
    def __len__(self): return len(self.cats)
    def __getitem__(self,i):
        if self.target is None: return self.cats[i],self.cont[i]
        return self.cats[i],self.cont[i],self.target[i]

class EmbedOrdinalMLP(nn.Module):
    def __init__(self, cards, n_cont):
        super().__init__()
        dims=[min(16,max(3,int(round((c**0.25)*4)))) for c in cards]
        self.emb=nn.ModuleList([nn.Embedding(c,d) for c,d in zip(cards,dims)])
        nin=sum(dims)+n_cont
        self.net=nn.Sequential(
            nn.Linear(nin,512),nn.BatchNorm1d(512),nn.GELU(),nn.Dropout(0.20),
            nn.Linear(512,256),nn.BatchNorm1d(256),nn.GELU(),nn.Dropout(0.15),
            nn.Linear(256,128),nn.GELU(),nn.Dropout(0.10),
        )
        self.reg=nn.Linear(128,1)
        self.ord=nn.Linear(128,3)
    def forward(self,cats,cont):
        es=[e(cats[:,j]) for j,e in enumerate(self.emb)]
        x=torch.cat(es+[cont],dim=1)
        h=self.net(x)
        return self.reg(h).squeeze(1),self.ord(h)

def fit_deep_preprocessor(fit_df):
    cat_cols=lowcard.copy(); cont_cols=[c for c in raw_features if c not in cat_cols]
    maps={}; cards=[]
    cats=np.zeros((len(fit_df),len(cat_cols)),dtype=np.int64)
    for j,c in enumerate(cat_cols):
        s=fit_df[c].where(fit_df[c].notna(),'__MISSING__').astype(str)
        vals=pd.Index(s.unique()); mp={v:i+1 for i,v in enumerate(vals)}
        maps[c]=mp; cats[:,j]=s.map(mp).fillna(0).astype(int); cards.append(len(mp)+1)
    if cont_cols:
        A=np.column_stack([pd.to_numeric(fit_df[c],errors='coerce').to_numpy(float) for c in cont_cols])
        med=np.nanmedian(A,axis=0); miss=np.isnan(A).astype(np.float32); inds=np.where(np.isnan(A)); A[inds]=np.take(med,inds[1])
        scaler=StandardScaler(); An=scaler.fit_transform(A).astype(np.float32)
        cont=np.hstack([An,miss]).astype(np.float32)
    else:
        med=np.array([]); scaler=None; cont=np.empty((len(fit_df),0),np.float32)
    return {'cat_cols':cat_cols,'cont_cols':cont_cols,'maps':maps,'cards':cards,'med':med,'scaler':scaler},cats,cont

def transform_deep(prep,df):
    cats=np.zeros((len(df),len(prep['cat_cols'])),dtype=np.int64)
    for j,c in enumerate(prep['cat_cols']):
        s=df[c].where(df[c].notna(),'__MISSING__').astype(str)
        cats[:,j]=s.map(prep['maps'][c]).fillna(0).astype(int)
    if prep['cont_cols']:
        A=np.column_stack([pd.to_numeric(df[c],errors='coerce').to_numpy(float) for c in prep['cont_cols']])
        miss=np.isnan(A).astype(np.float32); inds=np.where(np.isnan(A)); A[inds]=np.take(prep['med'],inds[1])
        An=prep['scaler'].transform(A).astype(np.float32); cont=np.hstack([An,miss]).astype(np.float32)
    else: cont=np.empty((len(df),0),np.float32)
    return cats,cont

def deep_score(model,loader,device):
    model.eval(); out=[]
    with torch.no_grad():
        for batch in loader:
            cats,cont=batch[:2]; cats=cats.to(device); cont=cont.to(device)
            reg,logits=model(cats,cont)
            ordexp=1.0+torch.sigmoid(logits).sum(1)
            score=0.5*reg+0.5*ordexp
            out.append(score.detach().cpu().numpy())
    return np.concatenate(out)

def train_deep_candidate(fit_df,yfit,cal_df,ycal,probe=True,seed=42):
    torch.manual_seed(seed); np.random.seed(seed)
    prep,cfit,nfit=fit_deep_preprocessor(fit_df)
    ccal,ncal=transform_deep(prep,cal_df)
    fit_loader=DataLoader(TabDataset(cfit,nfit,yfit),batch_size=CFG['DEEP_BATCH'],shuffle=True,num_workers=0,pin_memory=True)
    cal_loader=DataLoader(TabDataset(ccal,ncal),batch_size=CFG['DEEP_BATCH']*2,shuffle=False,num_workers=0)
    device=torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    model=EmbedOrdinalMLP(prep['cards'],nfit.shape[1]).to(device)
    opt=torch.optim.AdamW(model.parameters(),lr=1.5e-3,weight_decay=1e-4)
    mse=nn.MSELoss(); bce=nn.BCEWithLogitsLoss()
    scaler=torch.amp.GradScaler('cuda',enabled=(device.type=='cuda'))
    epochs=CFG['DEEP_EPOCHS_PROBE'] if probe else CFG['DEEP_EPOCHS_FULL']
    best_state=None; best=-999; stale=0
    for ep in range(epochs):
        model.train()
        for cats,cont,yy in fit_loader:
            cats,cont,yy=cats.to(device),cont.to(device),yy.to(device)
            ordy=torch.stack([(yy>1).float(),(yy>2).float(),(yy>3).float()],dim=1)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda',enabled=(device.type=='cuda')):
                reg,logits=model(cats,cont)
                loss=0.60*mse(reg,yy)+0.40*bce(logits,ordy)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        sc=deep_score(model,cal_loader,device); th=optimize_thresholds(ycal,sc); qq=qwk(ycal,apply_thresholds(sc,th))
        if qq>best+1e-4:
            best=qq; stale=0; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        else: stale+=1
        if stale>=CFG['DEEP_PATIENCE']: break
    if best_state is not None: model.load_state_dict(best_state)
    return model,prep,device

## 6. Modeling — Multi-Probe Kill Matrix

Each candidate is evaluated on **three untouched outer folds**. Within each outer-training partition, a separate inner calibration split is used for early stopping, threshold calibration, and candidate-vs-baseline blending. This prevents the V3 failure mode where a single favourable fold promoted weak ideas.

In [15]:
def prepare_baseline_frames():
    return build_tree_frame(Xraw),build_tree_frame(Traw)

BASE_X,BASE_T=prepare_baseline_frames()

probe_records=[]
probe_predictions={}

def eval_one_probe(fold_id, candidate):
    outer_tr,outer_va=folds[fold_id]
    fit_idx,cal_idx=inner_fit_cal_split(outer_tr,700+fold_id)
    yfit,ycal,yva=y[fit_idx],y[cal_idx],y[outer_va]

    # Common baseline, identical comparator for every candidate on this probe fold.
    base=fit_lgb(BASE_X.iloc[fit_idx],yfit,BASE_X.iloc[cal_idx],ycal,probe=True,seed=900+fold_id)
    bcal=base.predict(BASE_X.iloc[cal_idx]); bva=base.predict(BASE_X.iloc[outer_va])
    bth=optimize_thresholds(ycal,bcal); bq=qwk(yva,apply_thresholds(bva,bth)); bauc=boundary_auc(yva,bva)['auc_gt1']

    cand_cal=cand_va=None; meta={}
    if candidate=='catboost_allcat':
        trans=make_catboost_allcat_frames(fit_idx,cal_idx)
        xf=trans(Xraw.iloc[fit_idx]); xc=trans(Xraw.iloc[cal_idx]); xv=trans(Xraw.iloc[outer_va])
        m=fit_catboost_allcat(xf,yfit,xc,ycal,probe=True,seed=1000+fold_id)
        cand_cal=m.predict(xc); cand_va=m.predict(xv)

    elif candidate=='fingerprint_aug':
        xf=pd.concat([BASE_X.iloc[fit_idx].reset_index(drop=True),FPX.iloc[fit_idx].reset_index(drop=True)],axis=1)
        xc=pd.concat([BASE_X.iloc[cal_idx].reset_index(drop=True),FPX.iloc[cal_idx].reset_index(drop=True)],axis=1)
        xv=pd.concat([BASE_X.iloc[outer_va].reset_index(drop=True),FPX.iloc[outer_va].reset_index(drop=True)],axis=1)
        m=fit_lgb(xf,yfit,xc,ycal,probe=True,seed=1100+fold_id)
        cand_cal=m.predict(xc); cand_va=m.predict(xv)

    elif candidate=='retrieval':
        cat_sel,cont_sel=select_retrieval_features(fit_idx)
        ref_idx=fit_idx
        if len(ref_idx)>CFG['RETRIEVAL_MAX_REF_PROBE']:
            rng=np.random.default_rng(1200+fold_id)
            ref_idx=np.sort(rng.choice(ref_idx,CFG['RETRIEVAL_MAX_REF_PROBE'],replace=False))
        enc,Zref=fit_retrieval_encoder(Xraw.iloc[ref_idx],cat_sel,cont_sel)
        Zcal=transform_retrieval(enc,Xraw.iloc[cal_idx]); Zva=transform_retrieval(enc,Xraw.iloc[outer_va])
        cand_cal=retrieval_predict(Zref,y[ref_idx],Zcal,CFG['RETRIEVAL_K'])
        cand_va=retrieval_predict(Zref,y[ref_idx],Zva,CFG['RETRIEVAL_K'])
        meta={'cat_features':len(cat_sel),'cont_features':len(cont_sel),'ref_rows':len(ref_idx)}

    elif candidate=='deep_embed_multitask':
        m,prep,dev=train_deep_candidate(Xraw.iloc[fit_idx],yfit,Xraw.iloc[cal_idx],ycal,probe=True,seed=1300+fold_id)
        ccal,ncal=transform_deep(prep,Xraw.iloc[cal_idx]); cva,nva=transform_deep(prep,Xraw.iloc[outer_va])
        cand_cal=deep_score(m,DataLoader(TabDataset(ccal,ncal),batch_size=2048,shuffle=False),dev)
        cand_va=deep_score(m,DataLoader(TabDataset(cva,nva),batch_size=2048,shuffle=False),dev)
        del m; torch.cuda.empty_cache() if torch.cuda.is_available() else None

    else: raise ValueError(candidate)

    tune=tune_blend(ycal,bcal,cand_cal)
    blended_va=(1-tune['alpha'])*bva+tune['alpha']*cand_va
    pred=apply_thresholds(blended_va,tune['thresholds'])
    cq=qwk(yva,pred); cauc=boundary_auc(yva,blended_va)['auc_gt1']; crmse=mean_squared_error(yva,blended_va)**0.5
    standalone_th=optimize_thresholds(ycal,cand_cal)
    standalone_q=qwk(yva,apply_thresholds(cand_va,standalone_th))
    residual_corr=np.corrcoef(yva-bva,yva-cand_va)[0,1]
    rec={
        'candidate':candidate,'fold':fold_id,'baseline_qwk':bq,'candidate_blend_qwk':cq,
        'gain_qwk':cq-bq,'candidate_standalone_qwk':standalone_q,'baseline_auc_gt1':bauc,
        'candidate_auc_gt1':cauc,'gain_auc_gt1':cauc-bauc,'rmse':crmse,
        'blend_alpha':tune['alpha'],'residual_corr_vs_baseline':residual_corr,**meta
    }
    print(rec)
    return rec

CANDIDATES=['catboost_allcat','fingerprint_aug','retrieval','deep_embed_multitask']
for cand in CANDIDATES:
    print('\n'+'='*90+'\nCANDIDATE:',cand)
    for f in CFG['PROBE_FOLDS']:
        t=time.time(); rec=eval_one_probe(f,cand); rec['seconds']=time.time()-t; probe_records.append(rec)
        gc.collect()

probe_df=pd.DataFrame(probe_records)
probe_df.to_csv(OUT/'breakthrough_probe_results.csv',index=False)
print('\nProbe results:')
print(probe_df.to_string(index=False))


CANDIDATE: catboost_allcat
{'candidate': 'catboost_allcat', 'fold': 0, 'baseline_qwk': np.float64(0.4714930964623524), 'candidate_blend_qwk': np.float64(0.47350437324815875), 'gain_qwk': np.float64(0.0020112767858063707), 'candidate_standalone_qwk': np.float64(0.47419180421897733), 'baseline_auc_gt1': np.float64(0.776647146533493), 'candidate_auc_gt1': np.float64(0.7775274972648175), 'gain_auc_gt1': np.float64(0.0008803507313245307), 'rmse': 0.5115885390189341, 'blend_alpha': 0.55, 'residual_corr_vs_baseline': np.float64(0.9953015990442965)}
{'candidate': 'catboost_allcat', 'fold': 2, 'baseline_qwk': np.float64(0.4789051272114144), 'candidate_blend_qwk': np.float64(0.4851299556886116), 'gain_qwk': np.float64(0.006224828477197186), 'candidate_standalone_qwk': np.float64(0.48420820295416045), 'baseline_auc_gt1': np.float64(0.7785033060421495), 'candidate_auc_gt1': np.float64(0.779680209916279), 'gain_auc_gt1': np.float64(0.001176903874129498), 'rmse': 0.5174430111283052, 'blend_alpha': 

### 6.1 Promotion / kill decision

In [16]:
summary_rows=[]
for cand,g in probe_df.groupby('candidate'):
    mean_q=g['candidate_blend_qwk'].mean(); mean_gain=g['gain_qwk'].mean(); wins=int((g['gain_qwk']>0).sum())
    mean_auc_gain=g['gain_auc_gt1'].mean(); mean_rescorr=g['residual_corr_vs_baseline'].mean()
    promote=((mean_gain>=CFG['PROMOTE_MIN_MEAN_GAIN'] and wins>=CFG['PROMOTE_MIN_WINS']) or
             (mean_q>=CFG['PROMOTE_ABS_QWK'] and wins>=2))
    status='PROMOTE' if promote else ('KILL' if mean_gain<CFG['KILL_MAX_MEAN_GAIN'] else 'WATCH')
    summary_rows.append({'candidate':cand,'mean_qwk':mean_q,'mean_gain':mean_gain,'wins':wins,
                         'mean_auc_gt1_gain':mean_auc_gain,'mean_residual_corr':mean_rescorr,'status':status})
probe_summary=pd.DataFrame(summary_rows).sort_values(['status','mean_qwk'],ascending=[False,False])
print(probe_summary.to_string(index=False))
probe_summary.to_csv(OUT/'breakthrough_probe_summary.csv',index=False)

promoted=probe_summary.loc[probe_summary.status=='PROMOTE'].sort_values('mean_qwk',ascending=False)['candidate'].tolist()
BEST_CANDIDATE=promoted[0] if promoted else None
print('\nBEST_CANDIDATE =',BEST_CANDIDATE)
if BEST_CANDIDATE is None:
    print('No hypothesis showed a qualitative jump. Full CV will be skipped intentionally.')

           candidate  mean_qwk  mean_gain  wins  mean_auc_gt1_gain  mean_residual_corr status
     catboost_allcat  0.474873   0.002925     3           0.001034            0.995176   KILL
deep_embed_multitask  0.472883   0.000935     1           0.000433            0.978518   KILL
           retrieval  0.471771  -0.000177     1          -0.000006            0.947488   KILL
     fingerprint_aug  0.470889  -0.001059     1           0.000289            0.997724   KILL

BEST_CANDIDATE = None
No hypothesis showed a qualitative jump. Full CV will be skipped intentionally.


## 7. Evaluation — Conditional Full 5-Fold CV

Only the best promoted candidate is allowed into full CV. If `BEST_CANDIDATE is None`, this section performs no expensive model training. This is intentional.

In [17]:
def train_candidate_full(candidate, fit_idx, cal_idx, val_idx, fold_id, need_test=True):
    yfit,ycal=y[fit_idx],y[cal_idx]
    base=fit_lgb(BASE_X.iloc[fit_idx],yfit,BASE_X.iloc[cal_idx],ycal,probe=False,seed=2000+fold_id)
    bcal=base.predict(BASE_X.iloc[cal_idx]); bval=base.predict(BASE_X.iloc[val_idx]); btest=base.predict(BASE_T) if need_test else None

    if candidate=='catboost_allcat':
        trans=make_catboost_allcat_frames(fit_idx,cal_idx)
        xf,xc,xv,xt=trans(Xraw.iloc[fit_idx]),trans(Xraw.iloc[cal_idx]),trans(Xraw.iloc[val_idx]),trans(Traw)
        m=fit_catboost_allcat(xf,yfit,xc,ycal,probe=False,seed=2100+fold_id)
        ccal,cval,ctest=m.predict(xc),m.predict(xv),m.predict(xt)

    elif candidate=='fingerprint_aug':
        def aug(idx, source_base, source_fp):
            return pd.concat([source_base.iloc[idx].reset_index(drop=True),source_fp.iloc[idx].reset_index(drop=True)],axis=1)
        xf=aug(fit_idx,BASE_X,FPX); xc=aug(cal_idx,BASE_X,FPX); xv=aug(val_idx,BASE_X,FPX)
        xt=pd.concat([BASE_T.reset_index(drop=True),FPT.reset_index(drop=True)],axis=1)
        m=fit_lgb(xf,yfit,xc,ycal,probe=False,seed=2200+fold_id)
        ccal,cval,ctest=m.predict(xc),m.predict(xv),m.predict(xt)

    elif candidate=='retrieval':
        cat_sel,cont_sel=select_retrieval_features(fit_idx)
        ref_idx=fit_idx
        if len(ref_idx)>CFG['RETRIEVAL_MAX_REF_FULL']:
            rng=np.random.default_rng(2300+fold_id); ref_idx=np.sort(rng.choice(ref_idx,CFG['RETRIEVAL_MAX_REF_FULL'],replace=False))
        enc,Zref=fit_retrieval_encoder(Xraw.iloc[ref_idx],cat_sel,cont_sel)
        ccal=retrieval_predict(Zref,y[ref_idx],transform_retrieval(enc,Xraw.iloc[cal_idx]),CFG['RETRIEVAL_K'])
        cval=retrieval_predict(Zref,y[ref_idx],transform_retrieval(enc,Xraw.iloc[val_idx]),CFG['RETRIEVAL_K'])
        ctest=retrieval_predict(Zref,y[ref_idx],transform_retrieval(enc,Traw),CFG['RETRIEVAL_K'])

    elif candidate=='deep_embed_multitask':
        m,prep,dev=train_deep_candidate(Xraw.iloc[fit_idx],yfit,Xraw.iloc[cal_idx],ycal,probe=False,seed=2400+fold_id)
        def ds(df):
            cc,nnn=transform_deep(prep,df); return deep_score(m,DataLoader(TabDataset(cc,nnn),batch_size=2048,shuffle=False),dev)
        ccal,cval,ctest=ds(Xraw.iloc[cal_idx]),ds(Xraw.iloc[val_idx]),ds(Traw)
        del m; torch.cuda.empty_cache() if torch.cuda.is_available() else None
    else: raise ValueError(candidate)

    tune=tune_blend(ycal,bcal,ccal,alphas=np.linspace(0,1,41))
    vscore=(1-tune['alpha'])*bval+tune['alpha']*cval
    tscore=(1-tune['alpha'])*btest+tune['alpha']*ctest
    return vscore,tscore,tune

FULL_CV_RAN=False
full_oof=np.full(len(train),np.nan,float)
full_test_folds=[]
full_fold_records=[]

if BEST_CANDIDATE is not None:
    FULL_CV_RAN=True
    for f,(otr,ova) in enumerate(folds):
        print('\nFULL CV FOLD',f,'candidate',BEST_CANDIDATE)
        fit_idx,cal_idx=inner_fit_cal_split(otr,3000+f)
        vs,ts,tune=train_candidate_full(BEST_CANDIDATE,fit_idx,cal_idx,ova,f,True)
        full_oof[ova]=vs; full_test_folds.append(ts)
        vp=apply_thresholds(vs,tune['thresholds'])
        rec={'fold':f,'qwk':qwk(y[ova],vp),'alpha':tune['alpha'],'thresholds':tune['thresholds'].tolist(),**boundary_auc(y[ova],vs)}
        full_fold_records.append(rec); print(rec)
    full_test=np.mean(full_test_folds,axis=0)
else:
    full_test=None

### 7.1 Honest unique-ID score and 0.61541 submission gate

In [18]:
SUBMISSION_ELIGIBLE=False
full_metrics={}
if FULL_CV_RAN:
    assert np.isfinite(full_oof).all()
    # Each duplicate id is in one fold. Aggregate continuous OOF first, then use the fold-calibrated class proxy for honest reporting.
    tmp=pd.DataFrame({'id':train[ID_COL].to_numpy(),'y':y,'score':full_oof})
    uniq=tmp.groupby('id',sort=False).agg(y=('y','first'),score=('score','mean')).reset_index()

    # Honest cross-fitted thresholds: recompute thresholds using other folds only, then evaluate held-out fold.
    cross_pred=np.zeros(len(train),int)
    fold_assign=np.full(len(train),-1,int)
    for f,(_,va) in enumerate(folds): fold_assign[va]=f
    for f in range(CFG['N_FOLDS']):
        trm=fold_assign!=f; vam=fold_assign==f
        th=optimize_thresholds(y[trm],full_oof[trm])
        cross_pred[vam]=apply_thresholds(full_oof[vam],th)
    tmp['pred']=cross_pred
    uniqp=tmp.groupby('id',sort=False).agg(y=('y','first'),pred=('pred','first'),score=('score','mean')).reset_index()
    uq=qwk(uniqp.y,uniqp.pred)
    aucs=boundary_auc(uniqp.y.to_numpy(),uniqp.score.to_numpy())
    full_metrics={'unique_id_crossfit_qwk':uq,**aucs,'candidate':BEST_CANDIDATE}
    print('FULL honest metrics:',full_metrics)
    SUBMISSION_ELIGIBLE=bool(uq>=CFG['SUBMISSION_QWK_GATE'])
    print('SUBMISSION_ELIGIBLE =',SUBMISSION_ELIGIBLE)
else:
    print('No full CV candidate. Submission gate remains closed.')

No full CV candidate. Submission gate remains closed.


### 7.2 Submission generation only if the hard gate is cleared

In [19]:
submission_path=None
if SUBMISSION_ELIGIBLE:
    global_th=optimize_thresholds(y,full_oof)
    raw_test=pd.DataFrame({'id':test[ID_COL].to_numpy(),'score':full_test})
    unique_test=raw_test.groupby('id',sort=False,as_index=False)['score'].mean()
    unique_test['life_satisfaction']=apply_thresholds(unique_test['score'].to_numpy(),global_th).astype(int)
    submission=unique_test[['id','life_satisfaction']].copy()
    assert len(submission)==CFG['EXPECTED_UNIQUE_TEST_IDS']
    assert submission['id'].is_unique and submission.life_satisfaction.isin([1,2,3,4]).all()
    submission_path=OUT/'submission_v4_breakthrough.csv'
    submission.to_csv(submission_path,index=False)
    print('SUBMISSION CREATED:',submission_path)
else:
    print('NO SUBMISSION CREATED. This notebook will not spend a Kaggle slot below the 0.61541 internal gate.')

NO SUBMISSION CREATED. This notebook will not spend a Kaggle slot below the 0.61541 internal gate.


### 7.3 Save discovery artifacts safely

In [20]:
def json_default(obj):
    if isinstance(obj,np.generic): return obj.item()
    if isinstance(obj,np.ndarray): return obj.tolist()
    if isinstance(obj,Path): return str(obj)
    if isinstance(obj,set): return list(obj)
    if hasattr(obj,'item'):
        try:return obj.item()
        except Exception:pass
    if hasattr(obj,'tolist'):
        try:return obj.tolist()
        except Exception:pass
    return str(obj)

summary={
    'config':CFG,
    'probe_summary':probe_summary.to_dict(orient='records'),
    'best_candidate':BEST_CANDIDATE,
    'full_cv_ran':FULL_CV_RAN,
    'full_metrics':full_metrics,
    'submission_eligible':SUBMISSION_ELIGIBLE,
    'submission_path':submission_path,
}
with open(OUT/'v4_breakthrough_summary.json','w') as f:
    json.dump(summary,f,indent=2,default=json_default)
if FULL_CV_RAN:
    pd.DataFrame({'id':train[ID_COL],'y':y,'score':full_oof}).to_csv(OUT/'oof_v4_breakthrough.csv',index=False)
    pd.DataFrame(full_fold_records).to_csv(OUT/'full_cv_fold_metrics.csv',index=False)
    pd.DataFrame({'row':np.arange(len(test)),'id':test[ID_COL],'score':full_test}).to_csv(OUT/'test_scores_v4_breakthrough.csv',index=False)
print('Artifacts saved to',OUT)

Artifacts saved to /kaggle/working/oqdq5_v4_discovery


## Interpretation Guide

### What counts as a useful result?

- **KILL**: mean probe gain `< +0.005`. This is incremental noise territory relative to the 0.133 leaderboard gap.
- **WATCH**: some gain, but not enough consistency or magnitude to justify full CV yet.
- **PROMOTE**: average gain `>= +0.015` and wins all three probe folds, or absolute probe QWK `>= 0.520` with at least two wins.
- **SUBMISSION_ELIGIBLE**: final honest unique-ID OOF QWK `>= 0.61541`.

### Why the gate is intentionally severe

V1–V3 already demonstrated that conventional GBDT/feature-engineering changes live in roughly the 0.001–0.003 improvement regime. A current public score of 0.61541 cannot plausibly be explained by another tiny tuning improvement. This notebook is therefore designed to answer a different question: **does any genuinely different representation produce a qualitative jump?**

If all four candidates are killed, do **not** convert this notebook into another ensemble marathon. The next step should be another discovery cycle aimed at a new hypothesis class, not more tuning of the dead ones.